In [ ]:
import json
from collections import defaultdict

def data_partition(data, last_n):
    user_train = {}
    user_valid = {}
    user_test = {}

    total_users = len(data)

    # 나머지 사용자들에 대해 훈련/검증/테스트 데이터를 나눔
    for user_id, user_items in enumerate(data, 1):
        nfeedback = len(user_items)
        if nfeedback < 3:
            user_train[user_id] = user_items
            user_valid[user_id] = []
            user_test[user_id] = []
        else:
            user_train[user_id] = user_items[:-2]  # 마지막 2개 제외한 아이템은 훈련 데이터
            user_valid[user_id] = [user_items[-2]]  # 마지막 2번째 아이템은 검증 데이터
            user_test[user_id] = [user_items[-1]]   # 마지막 아이템은 테스트 데이터



    return [user_train, user_valid, user_test]


In [ ]:
def make_ml_1m_data(ratings):
  User = defaultdict(list)
  user_set = set()
  item_set = set()
  user_num = 0
  item_num = 0
  for rating in ratings:
    u, i, r, t = rating.strip().split("::")
    u, i = int(u), int(i)
    user_set.add(u)
    item_set.add(i)
    if(user_num <= u):
      user_num = u
    if(item_num <= i):
      item_num = i
    User[u].append(i)

  # 각 사용자의 상호작용 리스트를 시간 순으로 정렬
  for user_id in User:
    User[user_id].sort() # Assuming ratings are in time order, otherwise you need to sort by timestamp

  # data_partition 함수에 넣을 수 있도록 데이터 포맷을 만듭니다.
  # data_partition 함수는 사용자의 인덱스를 기준으로 작동하므로,
  # User 딕셔너리의 키(사용자 ID)를 순서대로 정렬하여 리스트로 만듭니다.
  # 그리고 각 사용자의 아이템 리스트만 추출합니다.
  sorted_user_ids = sorted(User.keys())
  data_for_partition = [User[user_id] for user_id in sorted_user_ids]

  return data_for_partition, user_num, item_num

# ratings.dat 파일을 읽어서 make_ml_1m_data 함수에 전달
with open('ratings.dat', 'r') as f:
    ratings_data = f.readlines()

data_for_partition, user_count, item_count = make_ml_1m_data(ratings_data)

# data_partition 함수 호출 예시
user_train, user_valid, user_test = data_partition(data_for_partition, last_n=1) # last_n 매개변수는 data_partition 함수 본문에서 사용되지 않지만 인자로 받도록 되어있으므로 넣어줍니다.

print("Number of users:", user_count)
print("Number of items:", item_count)

print("User Train (first few):", dict(list(user_train.items())[:5]))
print("User Valid (first few):", dict(list(user_valid.items())[:5]))
print("User Test (first few):", dict(list(user_test.items())[:5]))


Number of users: 6040
Number of items: 3952
User Train (first few): {1: [1, 48, 150, 260, 527, 531, 588, 594, 595, 608, 661, 720, 745, 783, 914, 919, 938, 1022, 1028, 1029, 1035, 1097, 1193, 1197, 1207, 1246, 1270, 1287, 1545, 1566, 1721, 1836, 1907, 1961, 1962, 2018, 2028, 2294, 2321, 2340, 2355, 2398, 2687, 2692, 2762, 2791, 2797, 2804, 2918, 3105, 3114], 2: [21, 95, 110, 163, 165, 235, 265, 292, 318, 349, 356, 368, 380, 434, 442, 457, 459, 480, 498, 515, 589, 590, 593, 647, 648, 736, 780, 902, 920, 982, 1084, 1090, 1096, 1103, 1124, 1188, 1193, 1196, 1198, 1207, 1210, 1213, 1217, 1225, 1244, 1245, 1246, 1247, 1253, 1259, 1265, 1293, 1357, 1370, 1372, 1385, 1408, 1442, 1527, 1537, 1544, 1552, 1597, 1610, 1687, 1690, 1784, 1792, 1801, 1834, 1873, 1917, 1945, 1953, 1954, 1955, 1957, 1962, 1968, 2002, 2006, 2028, 2067, 2126, 2194, 2236, 2268, 2278, 2312, 2321, 2353, 2359, 2396, 2427, 2490, 2501, 2571, 2628, 2717, 2728, 2852, 2858, 2881, 2916, 2943, 3030, 3035, 3068, 3071, 3095, 3105, 31

In [ ]:
import torch
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader

# 하이퍼파라미터 설정
batch_size = 128
maxlen = 100  # 시퀀스의 최대 길이
SEED = 42  # 난수 시드

# 모델, 손실 함수, 옵티마이저 등 설정
class Args:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.hidden_units = 50
        self.num_blocks = 2
        self.num_heads = 1
        self.dropout_rate = 0.2
        self.maxlen = 100  # 시퀀스의 최대 길이

args = Args()

In [ ]:
args.device

device(type='cuda')

In [ ]:
import torch
import numpy as np
import copy
import random

def evaluate_pytorch(model, dataset, args, is_test=True):
    model.eval()
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)

    NDCG = 0.0
    HT = 0.0
    valid_user = 0.0

    users = range(1, usernum + 1)
    if usernum > 10000:
        users = random.sample(users, 10000)

    for u in users:
        if len(train[u]) < 1:
            continue
        if is_test and len(test[u]) < 1:
            continue
        if not is_test and len(valid[u]) < 1:
            continue

        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        if is_test:
            seq[idx] = valid[u][0]
            idx -= 1
        for i in reversed(train[u]):
            seq[idx] = i
            idx -= 1
            if idx == -1:
                break

        rated = set(train[u])
        rated.add(0)
        ground_truth = test[u][0] if is_test else valid[u][0]
        item_idx = [ground_truth]

        while len(item_idx) < 101:
            t = np.random.randint(1, itemnum + 1)
            if t not in rated:
                item_idx.append(t)

        seq = torch.tensor(seq, dtype=torch.long, device=args.device).unsqueeze(0)
        item_idx = torch.tensor(item_idx, dtype=torch.long, device=args.device).unsqueeze(0)
        user = torch.tensor([u], dtype=torch.long, device=args.device)

        with torch.no_grad():
            logits = model.predict(user, seq, item_idx)
            predictions = -logits.cpu().numpy()[0]

        rank = predictions.argsort().argsort()[0]

        valid_user += 1

        if rank < 10:
            NDCG += 1 / np.log2(rank + 2)
            HT += 1

    model.train()
    return NDCG / valid_user, HT / valid_user


In [ ]:
import numpy as np
def random_neq(l,r,s):
  t = np.random.randint(l,r)
  while t in s:
    t = np.random.randint(l,r)
  return t

In [ ]:
import torch
import torch.optim as optim
import numpy as np
def sample(user_train, usernum, itemnum):
    user = np.random.randint(1, usernum+1)
    while len(user_train[user]) <= 1:
        user = np.random.randint(1, usernum+1)
    seq = np.zeros([maxlen], dtype=np.int32)
    pos = np.zeros([maxlen], dtype=np.int32)
    neg = np.zeros([maxlen], dtype=np.int32)
    idx = maxlen - 1
    nxt = user_train[user][-1]

    ts = set(user_train[user])
    for i in reversed(user_train[user][:-1]):
        seq[idx] = i
        pos[idx] = nxt
        if nxt != 0:
            neg[idx] = random_neq(1, itemnum+1, ts)
        nxt = i
        idx -= 1
        if idx == -1:
            break
    return (user, seq, pos, neg)

In [ ]:
def sample_batch(user_train, usernum, itemnum,batch_size=64):
  batch_user = []
  batch_seq = []
  batch_pos = []
  batch_neg = []

  for _ in range(batch_size):
    u, s, p, n = sample(user_train, usernum, itemnum)
    batch_user.append(u)
    batch_seq.append(s)
    batch_pos.append(p)
    batch_neg.append(n)
  return (np.array(batch_user),
          np.stack(batch_seq),
          np.stack(batch_pos),
          np.stack(batch_neg))

In [ ]:
import numpy as np
import torch


class PointWiseFeedForward(torch.nn.Module):
    def __init__(self, hidden_units, dropout_rate):

        super(PointWiseFeedForward, self).__init__()

        self.conv1 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout1 = torch.nn.Dropout(p=dropout_rate)
        self.relu = torch.nn.ReLU()
        self.conv2 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout2 = torch.nn.Dropout(p=dropout_rate)

    def forward(self, inputs):
        outputs = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(inputs.transpose(-1, -2))))))
        outputs = outputs.transpose(-1, -2) # as Conv1D requires (N, C, Length)
        outputs += inputs
        return outputs

In [ ]:
class SASRec(torch.nn.Module):
    def __init__(self, user_num, item_num, args):
        super(SASRec, self).__init__()

        self.user_num = user_num
        self.item_num = item_num
        self.dev = args.device

        self.item_emb = torch.nn.Embedding(self.item_num+1, args.hidden_units, padding_idx=0)
        self.pos_emb = torch.nn.Embedding(args.maxlen+1, args.hidden_units, padding_idx=0)
        self.emb_dropout = torch.nn.Dropout(p=args.dropout_rate)

        self.attention_layernorms = torch.nn.ModuleList() # to be Q for self-attention
        self.attention_layers = torch.nn.ModuleList()
        self.forward_layernorms = torch.nn.ModuleList()
        self.forward_layers = torch.nn.ModuleList()

        self.last_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)

        for _ in range(args.num_blocks):
            new_attn_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.attention_layernorms.append(new_attn_layernorm)

            new_attn_layer =  torch.nn.MultiheadAttention(args.hidden_units,
                                                            args.num_heads,
                                                            args.dropout_rate)
            self.attention_layers.append(new_attn_layer)

            new_fwd_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.forward_layernorms.append(new_fwd_layernorm)

            new_fwd_layer = PointWiseFeedForward(args.hidden_units, args.dropout_rate)
            self.forward_layers.append(new_fwd_layer)

    def log2feats(self, log_seqs): # TODO: fp64 and int64 as default in python, trim?

        log_seqs = log_seqs.to(torch.long).to(self.dev)

        seqs = self.item_emb(log_seqs)

        seqs *= self.item_emb.embedding_dim ** 0.5
        poss = torch.tensor(np.tile(np.arange(1, log_seqs.shape[1] + 1), [log_seqs.shape[0], 1]), dtype=torch.long).to(self.dev)
        poss *= (torch.tensor(log_seqs != 0, dtype=torch.long).to(self.dev))


        seqs += self.pos_emb(torch.tensor(poss,dtype=torch.long).to(self.dev))
        seqs = self.emb_dropout(seqs)

        tl = seqs.shape[1] # time dim len for enforce causality
        attention_mask = ~torch.tril(torch.ones((tl, tl), dtype=torch.bool, device=self.dev))

        for i in range(len(self.attention_layers)):
            seqs = torch.transpose(seqs, 0, 1)
            Q = self.attention_layernorms[i](seqs)
            mha_outputs, _ = self.attention_layers[i](Q, seqs, seqs,
                                            attn_mask=attention_mask)
                                            # need_weights=False) this arg do not work?
            seqs = Q + mha_outputs
            seqs = torch.transpose(seqs, 0, 1)

            seqs = self.forward_layernorms[i](seqs)
            seqs = self.forward_layers[i](seqs)

        log_feats = self.last_layernorm(seqs) # (U, T, C) -> (U, -1, C)

        return log_feats

    def forward(self, user_ids, log_seqs, pos_seqs, neg_seqs): # for training

        log_feats = self.log2feats(log_seqs) # user_ids hasn't been used yet

        pos_embs = self.item_emb(torch.tensor(pos_seqs,dtype=torch.long).to(self.dev))
        neg_embs = self.item_emb(torch.tensor(neg_seqs,dtype=torch.long).to(self.dev))

        pos_logits = (log_feats * pos_embs).sum(dim=-1)
        neg_logits = (log_feats * neg_embs).sum(dim=-1)

        # pos_pred = self.pos_sigmoid(pos_logits)
        # neg_pred = self.neg_sigmoid(neg_logits)

        return pos_logits, neg_logits # pos_pred, neg_pred

    def predict(self, user_ids, log_seqs, item_indices): # for inference
        log_feats = self.log2feats(log_seqs) # user_ids hasn't been used yet

        final_feat = log_feats[:, -1, :] # only use last QKV classifier, a waste

        item_embs = self.item_emb(torch.tensor(item_indices,dtype=torch.long).to(self.dev)) # (U, I, C)

        logits = item_embs.matmul(final_feat.unsqueeze(-1)).squeeze(-1)

        # preds = self.pos_sigmoid(logits) # rank same item list for different users

        return logits # preds # (U, I)

# origin_data

In [ ]:
import time
model = SASRec(user_count, item_count, args).to(args.device)
optimizer = optim.Adam(model.parameters(), lr=0.001, betas = (0.9, 0.98))
model.pos_emb.weight.data[0, :] = 0
model.item_emb.weight.data[0, :] = 0
criterion = torch.nn.BCEWithLogitsLoss()
epochs = 200
batch_size = 64
num_batches = user_count//batch_size  # 한 에폭 당 실행할 배치 수
model.train()

best_val_ndcg, best_val_hr = 0.0, 0.0
best_test_ndcg, best_test_hr = 0.0, 0.0
T = 0.0
t0 = time.time()

origin_losses, origin_val_ndcgs, origin_val_hrs, origin_test_ndcgs, origin_test_hrs = [], [], [], [], []

for epoch in range(epochs):

    total_loss = 0.0
    for i in range(num_batches):
        # 배치 샘플링
        user_batch, seq_batch, pos_batch, neg_batch = sample_batch(user_train,user_count, item_count,batch_size)
        indices = np.where(pos_batch!=0)
        # 텐서로 변환 후 디바이스에 올리기
        user_batch = torch.tensor(user_batch, dtype=torch.long, device=args.device)
        seq_batch = torch.tensor(seq_batch, dtype=torch.long, device=args.device)
        pos_batch = torch.tensor(pos_batch, dtype=torch.long, device=args.device)
        neg_batch = torch.tensor(neg_batch, dtype=torch.long, device=args.device)

        optimizer.zero_grad()

        # 모델 forward: 양성, 음성 로그잇 추출
        pos_logits, neg_logits = model(user_batch, seq_batch, pos_batch, neg_batch)

        # 타겟 라벨 생성: 양성은 1, 음성은 0
        pos_labels = torch.ones_like(pos_logits)
        neg_labels = torch.zeros_like(neg_logits)


        # 손실 계산: 양성과 음성 손실을 합산
        loss_pos = criterion(pos_logits, pos_labels)
        loss_neg = criterion(neg_logits, neg_labels)
        loss = loss_pos + loss_neg

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if(i % 100 == 1):
          print(i,"번째")

    avg_loss = total_loss / num_batches
    val_NDCG, val_HR = evaluate_pytorch(model, [user_train, user_valid, user_test, user_count, item_count], args, is_test=False)
    test_NDCG, test_HR = evaluate_pytorch(model, [user_train,user_valid, user_test,user_count,item_count], args, is_test=True)

    origin_losses.append(avg_loss)
    origin_val_ndcgs.append(val_NDCG)
    origin_val_hrs.append(val_HR)
    origin_test_ndcgs.append(test_NDCG)
    origin_test_hrs.append(test_HR)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, "
      f"Val NDCG: {val_NDCG:.4f}, Val HR: {val_HR:.4f}, "
      f"Test NDCG: {test_NDCG:.4f}, Test HR: {test_HR:.4f}")



<ipython-input-10-9df9bc1ba0ad>:43: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  poss *= (torch.tensor(log_seqs != 0, dtype=torch.long).to(self.dev))
<ipython-input-10-9df9bc1ba0ad>:46: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  seqs += self.pos_emb(torch.tensor(poss,dtype=torch.long).to(self.dev))
<ipython-input-10-9df9bc1ba0ad>:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_embs = self.item_emb(torch.tensor(pos_seqs,dtype=torch.long).to(self.dev))
<ipython-input-10-9df9bc1ba0ad>:73: UserWarning: To copy construct from a tenso

1 번째


<ipython-input-10-9df9bc1ba0ad>:88: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item_embs = self.item_emb(torch.tensor(item_indices,dtype=torch.long).to(self.dev)) # (U, I, C)


Epoch 1/200, Loss: 3.6064, Val NDCG: 0.0557, Val HR: 0.1300, Test NDCG: 0.0572, Test HR: 0.1419
1 번째
Epoch 2/200, Loss: 2.1960, Val NDCG: 0.0801, Val HR: 0.1821, Test NDCG: 0.0870, Test HR: 0.2051
1 번째
Epoch 3/200, Loss: 1.4690, Val NDCG: 0.1317, Val HR: 0.3179, Test NDCG: 0.1246, Test HR: 0.3015
1 번째
Epoch 4/200, Loss: 1.1727, Val NDCG: 0.2127, Val HR: 0.4526, Test NDCG: 0.1637, Test HR: 0.3586
1 번째
Epoch 5/200, Loss: 1.0587, Val NDCG: 0.2643, Val HR: 0.5061, Test NDCG: 0.2006, Test HR: 0.3886
1 번째
Epoch 6/200, Loss: 1.0307, Val NDCG: 0.2806, Val HR: 0.5202, Test NDCG: 0.2066, Test HR: 0.3935
1 번째
Epoch 7/200, Loss: 1.0173, Val NDCG: 0.2777, Val HR: 0.5156, Test NDCG: 0.2065, Test HR: 0.3899
1 번째
Epoch 8/200, Loss: 1.0041, Val NDCG: 0.2714, Val HR: 0.5066, Test NDCG: 0.2044, Test HR: 0.3874
1 번째
Epoch 9/200, Loss: 1.0028, Val NDCG: 0.2786, Val HR: 0.5210, Test NDCG: 0.2086, Test HR: 0.4036
1 번째
Epoch 10/200, Loss: 1.0024, Val NDCG: 0.2856, Val HR: 0.5310, Test NDCG: 0.2134, Test HR: 0

In [ ]:
torch.save(model.state_dict(), 'sasrec_model_weights.pth')
